# Criteria Check for DESEQ

Quality Assurance before Differential Expression Analysis

Criteria
- [x] Verify that expression and metadata files only contain valid series and samples currated in `/labeling`
- [x] Ensure that samples in the expression data are subsets of samples identified `/labeling`

# Getting Started
______

##### Load libraries

In [1]:
import pandas as pd
import os

#### Initialize variables

In [2]:
# Labeling paths
TBlabels =	pd.read_csv("../data/RNAseq_data_forDE/clean_TB_sample_metadata_classification.tsv", sep = "\t")
TBmicrolabels = pd.read_csv("../data/microarray_data_forDE/clean_TB_sample_metadata_classification.tsv", sep = "\t")

# General Paths 
TB_expression_path = '../data/RNAseq_data_forDE/expression_matrices/cleaned'
TB_metadata_path = '../data/RNAseq_data_forDE/metadata'

TBmicro_expression_path = '../data/microarray_data_forDE/expression_matrices/cleaned'
TBmicro_metadata_path = '../data/microarray_data_forDE/metadata'


In [3]:
def print_gse_gsm_counts(df, name):
	print(f"{name}:")
	# check if there are sample duplicates
	duplicates = df[df.duplicated(subset=['geo_accession'], keep=False)]
	if not duplicates.empty:
		print("  Warning: There are duplicate geo_accession entries")
		print(duplicates)
	else:
		print("  No duplicate geo_accession entries found")
	print(f"  Number of unique Series: {df['series_id'].nunique()}")
	print(f"  Number of unique Samples: {df['geo_accession'].nunique()}")
	print()

print_gse_gsm_counts(TBlabels, "TBlabels")
print_gse_gsm_counts(TBmicrolabels, "TBmicrolabels")


TBlabels:
  No duplicate geo_accession entries found
  Number of unique Series: 22
  Number of unique Samples: 515

TBmicrolabels:
  No duplicate geo_accession entries found
  Number of unique Series: 10
  Number of unique Samples: 723



# Help Functions

##### Check for Classification Criteria Fullfillment

* studies with only `disease without treatment` or only `healthy control without treatment` will be removed

* studies with less than 3 `disease without treatment` and 3 `healthy control without treatment` will be removed


In [4]:
def filter_studies_by_classification(df, drop=False, return_studies=False):
	# Define required classifications
	required_classes = {"healthy control without treatment", "disease without treatment"}
	unique_classes = set(df['CLASSIFICATION'].unique())
	
	# Check if only the two required classifications are present
	if unique_classes != required_classes:
		print(f"Error: classification column contains values other than {required_classes}: {unique_classes}")
		return
	
	# Group by series_id and classification, count samples
	counts = df.groupby(['series_id', 'CLASSIFICATION'])['geo_accession'].count().unstack(fill_value=0)
	counts.columns.name = 'classification'  # rename for display
	counts.rename_axis(columns=None, inplace=True)
	
	# Find studies with only one classification
	single_class_studies = counts[(counts == 0).any(axis=1)]
	# Find studies that have both classifications but not enough samples in each
	insufficient_class_studies = counts[
		(counts > 0).all(axis=1) & ~(counts >= 3).all(axis=1)
	]
	# Find studies with at least 3 of each classification
	valid_studies = counts[(counts >= 3).all(axis=1)]
	
	if single_class_studies.empty and insufficient_class_studies.empty:
		print("All studies are valid.")
	else:
		if not single_class_studies.empty:
			print("Studies with only one classification (excluded):")
			print(single_class_studies)
		if not insufficient_class_studies.empty:
			print("\nStudies dropped due to insufficient samples in one or both classifications:")
			print(insufficient_class_studies)
		print("\nStudies passing the criteria (at least 3 of each classification):")
		print(valid_studies)
	
	# Optionally drop studies that do not pass the criteria
	if drop:
		filtered_df = df[df['series_id'].isin(valid_studies.index)]
		return filtered_df, valid_studies
	if return_studies:
		return valid_studies, df['series_id'].unique().tolist()


##### Check for Samples Criteria Fullfillment

In [5]:
def check_expression_samples_per_study(expression_dir, df):
	for gse_id in df['series_id'].unique():
		expr_file = os.path.join(expression_dir, f"{gse_id}.tsv")
		if not os.path.exists(expr_file):
			print(f"Expression file not found for {gse_id}: {expr_file}")
			continue
		expr = pd.read_csv(expr_file, sep = "\t" , engine='python')
		expr_samples = set(expr.columns[1:])
		label_samples = set(df[df['series_id'] == gse_id]['geo_accession'])
		if expr_samples.issubset(label_samples):
			print(f"{gse_id}: Passed!")
		else:
			extra = expr_samples - label_samples
			print(f"{gse_id}: Extra samples in expression data not in labels: {extra}")

# TB criteria check

In [17]:
TBlabels.columns

Index(['organism_ch1', 'series_id', 'geo_accession', 'CLASSIFICATION', 'title',
       'SAMPLE_TYPE', 'CELL', 'TISSUE', 'TIME_POINT', 'SAMPLE_CONDITION',
       'source_name_ch1', 'characteristics_ch1', 'characteristics_ch1.1',
       'characteristics_ch1.2', 'molecule_ch1', 'platform_id',
       'library_source', 'library_strategy', 'cell.type.ch1', 'tissue.ch1',
       'time.ch1', 'CONTRAST', 'SIGNATURE_NAME', 'TB', 'EXPRMAT_PATH'],
      dtype='object')

In [6]:
filter_studies_by_classification(TBlabels, drop=False, return_studies=False)

All studies are valid.


In [7]:
filter_studies_by_classification(TBmicrolabels, drop=False, return_studies=False)

All studies are valid.


In [8]:
check_expression_samples_per_study(TB_expression_path, TBlabels)

GSE84076: Passed!
GSE148171: Passed!
GSE198557: Passed!
GSE112482: Passed!
GSE112483: Passed!
GSE132283: Passed!
GSE143627: Passed!
GSE148731: Passed!
GSE174566: Passed!
GSE193777: Passed!
GSE223863: Passed!
GSE256184: Passed!
GSE64179: Passed!
GSE64182: Passed!
GSE121049: Passed!
GSE141656: Passed!
GSE143731: Passed!
GSE164287: Passed!
GSE165708: Passed!
GSE183912: Passed!
GSE211974: Passed!
GSE236156: Passed!


In [9]:
check_expression_samples_per_study(TBmicro_expression_path, TBmicrolabels)

GSE16250: Passed!
GSE18794: Passed!
GSE19491: Passed!
GSE34151: Passed!
GSE42830: Passed!
GSE42834: Passed!
GSE63548: Passed!
GSE83456: Passed!
GSE119143: Passed!
GSE139871: Passed!
